In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import time
import os


In [2]:
# ==========================================
# 1. Dummy Data Generation
# ==========================================
def generate_dummy_csv(filename="transactions.csv", n_rows=10000):
    print(f"Generating dummy data: {filename}...")
    
    # Create somewhat realistic patterns
    n_accounts = 500
    account_ids = [f"ACC_{i:04d}" for i in range(n_accounts)]
    types = ["PAYMENT", "TRANSFER", "DEBIT", "FEE", "REFUND"]
    
    data = {
        'originator_account': np.random.choice(account_ids, n_rows),
        'beneficiary_account': np.random.choice(account_ids, n_rows),
        'timestamp': [time.time() - np.random.randint(0, 1000000) for _ in range(n_rows)],
        'amount': np.random.exponential(scale=100, size=n_rows).round(2),
        'type': np.random.choice(types, n_rows)
    }
    
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)
    print("Data generation complete.\n")


In [ ]:

# ==========================================
# 2. Data Preprocessing & Vocabulary
# ==========================================
class DataPreprocessor:
    def __init__(self):
        self.acct_encoder = LabelEncoder()
        self.type_encoder = LabelEncoder()
        self.scaler = StandardScaler()
        
    def fit_transform(self, df):
        print("Preprocessing data...")
        df = df.copy()
        
        # 1. Categorical Encoding (Accounts)
        all_accts = pd.concat([df['originator_account'], df['beneficiary_account']]).unique()
        self.acct_encoder.fit(all_accts)
        df['originator_idx'] = self.acct_encoder.transform(df['originator_account']) + 1
        df['beneficiary_idx'] = self.acct_encoder.transform(df['beneficiary_account']) + 1
        
        # 2. Categorical Encoding (Type)
        df['type_idx'] = self.type_encoder.fit_transform(df['type']) + 1
        
        # 3. Time Feature Engineering (Cyclical hour-of-day)
        dates = pd.to_datetime(df['timestamp'], unit='s')
        hours = dates.dt.hour
        df['time_sin'] = np.sin(2 * np.pi * hours / 24)
        df['time_cos'] = np.cos(2 * np.pi * hours / 24)
        
        # 4. Numerical Normalization (Amount)
        df[['amount']] = self.scaler.fit_transform(df[['amount']])
        
        # 5. Relative Time Delta (FraudTransformer-style, event-level per originator)
        # Sort by account + time, compute inter-event gap in seconds for each account.
        # First transaction per account gets delta=0.
        df = df.sort_values(['originator_account', 'timestamp']).reset_index(drop=True)
        df['time_delta'] = df.groupby('originator_account')['timestamp'].diff().fillna(0)
        
        self.vocab_size_acct = len(self.acct_encoder.classes_) + 2
        self.vocab_size_type = len(self.type_encoder.classes_) + 2
        
        return df


In [ ]:

# ==========================================
# 2b. Sinusoidal Time Encoder (FraudTransformer, Aminian et al. 2025)
# ==========================================
class SinusoidalTimeEncoder(nn.Module):
    """
    Encodes a scalar inter-event time delta into a d_emb-dimensional vector
    using GPT-style Fourier features (sinusoidal positional encoding), then
    applies LayerNorm and scales by a learnable gain lambda.

    From FraudTransformer (SRP variant):
        theta_i  = f_base ^ (-2i / d_emb)
        e_t      = [ sin(theta * id_t) ; cos(theta * id_t) ]  in R^d_emb
        output   = lambda * LayerNorm(e_t)
    """
    def __init__(self, d_emb, f_base=1e7):
        super().__init__()
        assert d_emb % 2 == 0, "d_emb must be even for sinusoidal encoding"
        i = torch.arange(0, d_emb // 2, dtype=torch.float32)
        theta = f_base ** (-2.0 * i / d_emb)          # (d_emb//2,)
        self.register_buffer('theta', theta)
        self.layer_norm = nn.LayerNorm(d_emb)
        self.lam = nn.Parameter(torch.tensor(0.01))    # learnable time gain

    def forward(self, time_delta):
        # time_delta: (batch,) float32 — inter-event seconds
        angles = time_delta.unsqueeze(1) * self.theta.unsqueeze(0)  # (B, d_emb//2)
        e = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)  # (B, d_emb)
        return self.lam * self.layer_norm(e)


In [ ]:

# ==========================================
# 3. Contrastive Dataset
# ==========================================
class ContrastiveTransactionDataset(Dataset):
    def __init__(self, processed_df):
        self.data = processed_df
        self.orig = torch.tensor(self.data['originator_idx'].values, dtype=torch.long)
        self.bene = torch.tensor(self.data['beneficiary_idx'].values, dtype=torch.long)
        self.txn_type = torch.tensor(self.data['type_idx'].values, dtype=torch.long)
        self.numerical = torch.tensor(
            self.data[['amount', 'time_sin', 'time_cos']].values, 
            dtype=torch.float32
        )
        # Inter-event time delta — passed through unchanged (not augmented)
        self.time_delta = torch.tensor(self.data['time_delta'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.data)
    
    def augment(self, orig, bene, txn_type, numerical):
        """Create a 'view' of the data by randomly corrupting features."""
        orig_aug = orig.clone()
        bene_aug = bene.clone()
        type_aug = txn_type.clone()
        num_aug = numerical.clone()
        
        mask_prob = 0.2
        if torch.rand(1) < mask_prob: orig_aug *= 0
        if torch.rand(1) < mask_prob: bene_aug *= 0
        if torch.rand(1) < mask_prob: type_aug *= 0
        
        noise = torch.randn_like(num_aug) * 0.1
        num_aug += noise
        
        return orig_aug, bene_aug, type_aug, num_aug

    def __getitem__(self, idx):
        o, b, t, n = self.orig[idx], self.bene[idx], self.txn_type[idx], self.numerical[idx]
        td = self.time_delta[idx]
        
        # Two augmented views; time_delta is the same for both (not augmented)
        view1 = self.augment(o, b, t, n) + (td,)
        view2 = self.augment(o, b, t, n) + (td,)
        
        return view1, view2


In [ ]:

# ==========================================
# 4. The Embedding Model
# ==========================================
class TransactionEncoder(nn.Module):
    def __init__(self, vocab_acct, vocab_type, embedding_dim=64):
        super(TransactionEncoder, self).__init__()
        
        # 1. Embedding Layers
        self.acct_embedding = nn.Embedding(vocab_acct, embedding_dim, padding_idx=0)
        self.type_embedding = nn.Embedding(vocab_type, embedding_dim // 2, padding_idx=0)
        
        # 2. Numerical Processor (MLP)
        self.num_mlp = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )
        
        # Orig(64) + Bene(64) + Type(32) + Numerical(32) = 192
        concat_dim = embedding_dim * 2 + (embedding_dim // 2) + 32
        
        # 3. Sinusoidal Time Encoder (FraudTransformer SRP variant)
        # Produces a concat_dim-sized additive correction to x, scaled by learnable lambda
        self.time_encoder = SinusoidalTimeEncoder(concat_dim)
        
        # 4. Main Encoder
        self.encoder = nn.Sequential(
            nn.Linear(concat_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        
        # 5. Projection Head (discarded after training)
        self.projector = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def forward_one_view(self, orig, bene, txn_type, num, time_delta):
        emb_o = self.acct_embedding(orig)
        emb_b = self.acct_embedding(bene)
        emb_t = self.type_embedding(txn_type)
        emb_n = self.num_mlp(num)
        
        x = torch.cat([emb_o, emb_b, emb_t, emb_n], dim=1)  # (B, concat_dim)
        
        # Inject sinusoidal time encoding additively (FraudTransformer-style)
        x = x + self.time_encoder(time_delta)
        
        representation = self.encoder(x)
        projection = self.projector(representation)
        return representation, projection

    def forward(self, view1, view2):
        o1, b1, t1, n1, td1 = view1
        o2, b2, t2, n2, td2 = view2
        
        h1, z1 = self.forward_one_view(o1, b1, t1, n1, td1)
        h2, z2 = self.forward_one_view(o2, b2, t2, n2, td2)
        
        return z1, z2


In [6]:


# ==========================================
# 5. Contrastive Loss (NT-Xent)
# ==========================================
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]
        
        # Normalize vectors
        z_i = torch.nn.functional.normalize(z_i, dim=1)
        z_j = torch.nn.functional.normalize(z_j, dim=1)
        
        # Concatenate outputs
        representations = torch.cat([z_i, z_j], dim=0)
        
        # Similarity matrix (Cosine similarity)
        similarity_matrix = torch.matmul(representations, representations.T)
        
        # Create labels: For index i, the positive is i + batch_size
        # (and vice versa)
        labels = torch.cat([
            torch.arange(batch_size, 2 * batch_size), 
            torch.arange(batch_size)
        ], dim=0).to(z_i.device)
        
        # Mask out self-similarity (diagonal)
        mask = torch.eye(labels.shape[0], dtype=torch.bool).to(z_i.device)
        similarity_matrix = similarity_matrix / self.temperature
        
        # We need to remove the diagonal (self-similarity) from calculations
        # But CrossEntropyLoss expects standard logits. 
        # A simpler trick for SimCLR implementation:
        
        # Select positives
        sim_ij = torch.diag(similarity_matrix, batch_size)
        sim_ji = torch.diag(similarity_matrix, -batch_size)
        positives = torch.cat([sim_ij, sim_ji], dim=0).view(2 * batch_size, 1)
        
        # Select negatives (all other similarities)
        # This part requires careful masking, simplified here for readability:
        # In a production SimCLR, we compute full CrossEntropy against all pairs
        
        logits = similarity_matrix # (2N, 2N)
        # To strictly use CrossEntropy, we set diagonal to -infinity
        logits.fill_diagonal_(-float('inf'))
        
        return self.criterion(logits, labels)

In [ ]:

# ==========================================
# 6. Training Pipeline
# ==========================================
def train_embeddings():
    BATCH_SIZE = 256
    EPOCHS = 5
    LR = 3e-4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on device: {DEVICE}")

    if not os.path.exists("transactions.csv"):
        generate_dummy_csv()
        
    df = pd.read_csv("transactions.csv")
    preprocessor = DataPreprocessor()
    df_processed = preprocessor.fit_transform(df)
    
    dataset = ContrastiveTransactionDataset(df_processed)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    
    model = TransactionEncoder(
        vocab_acct=preprocessor.vocab_size_acct,
        vocab_type=preprocessor.vocab_size_type
    ).to(DEVICE)
    
    loss_fn = NTXentLoss(temperature=0.1)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    print("\nStarting training...")
    model.train()
    
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch in dataloader:
            view1_raw, view2_raw = batch
            view1 = [t.to(DEVICE) for t in view1_raw]
            view2 = [t.to(DEVICE) for t in view2_raw]
            
            optimizer.zero_grad()
            z1, z2 = model(view1, view2)
            loss = loss_fn(z1, z2)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss / len(dataloader):.4f}")

    # ==========================================
    # 7. Inference / Extraction
    # ==========================================
    print("\nExtracting embeddings for first 5 transactions...")
    model.eval()
    
    sample_data = df_processed.iloc[:5]
    
    orig = torch.tensor(sample_data['originator_idx'].values, dtype=torch.long).to(DEVICE)
    bene = torch.tensor(sample_data['beneficiary_idx'].values, dtype=torch.long).to(DEVICE)
    typ  = torch.tensor(sample_data['type_idx'].values, dtype=torch.long).to(DEVICE)
    num  = torch.tensor(sample_data[['amount', 'time_sin', 'time_cos']].values, dtype=torch.float32).to(DEVICE)
    td   = torch.tensor(sample_data['time_delta'].values, dtype=torch.float32).to(DEVICE)
    
    with torch.no_grad():
        embeddings, _ = model.forward_one_view(orig, bene, typ, num, td)
        
    print("Embedding Shape:", embeddings.shape)
    print("First Embedding Vector sample:", embeddings[0][:5])

if __name__ == "__main__":
    train_embeddings()
